# PMEAPI: Probabilistic Multi-Evidence Adversarial Patch Inference

In [1]:
# pmeapi.ipynb – first cell
import os, sys, torch
import warnings
warnings.filterwarnings("ignore")   # suppress any path-related warnings

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
sys.path.insert(0, os.path.abspath('.'))

from src.config import DEVICE, RESULTS_DIR, CHECKPOINT_DIR
from src.model import get_model
from src.data_loader import build_partition_indices, get_partition
from src.attack import load_patch, apply_patch
from src.gradcam_defense import GradCAM
from src.feature_squeeze import load_threshold

print(f"Device: {DEVICE}")
model        = get_model()
patch        = load_patch()
partitions   = build_partition_indices()
gradcam      = GradCAM(model, target_layer_name='layer4')
fs_threshold = load_threshold()
print(f"FS threshold: {fs_threshold:.3f}")

import torch
results = torch.load('results/results_with_ci.pt', weights_only=False)
print("Base results loaded:", list(results.keys()))

[config] Using device: mps
Device: mps
Loading pretrained ResNet-50 …
Ready
Patch loaded  shape: (3, 89, 89)
Threshold loaded: 0.150
FS threshold: 0.150
Base results loaded: ['no_defense', 'jpeg', 'feature_squeeze', 'gradcam_only', 'hybrid']


## Step 1: Load calibration and threshold-tuning data

Uses the `fs_tuning` partition (500 images, disjoint from `final_eval`).
- First half (0:250) → Beta calibration
- Second half (250:) → decision threshold tuning
These two slices are never swapped.

In [2]:

cal_clean, _ = get_partition("fs_tuning", partitions)
n_cal        = cal_clean.size(0) // 2

cal_clean_half   = cal_clean[:n_cal]
cal_patched_half, _ = apply_patch(cal_clean_half.clone(), patch, augment=False)

thresh_clean   = cal_clean[n_cal:]
thresh_patched, _ = apply_patch(thresh_clean.clone(), patch, augment=False)

print(f"Calibration set:      {cal_clean_half.size(0)} images  (fs_tuning[0:{n_cal}])")
print(f"Threshold-tuning set: {thresh_clean.size(0)} images  (fs_tuning[{n_cal}:])")


Calibration set:      250 images  (fs_tuning[0:250])
Threshold-tuning set: 250 images  (fs_tuning[250:])


## Step 2: Calibrate Beta sensor likelihoods

In [3]:

import os, torch
_path = 'results/checkpoints/pmeapi_params.pt'
if os.path.exists(_path):
    print("[SKIP] pmeapi_params.pt exists -- loading.")
    from src.probabilistic_fusion import load_calibration
    pmeapi_params = load_calibration()
    for s in ["orb", "fs", "gc"]:
        p = pmeapi_params[s]
        print(f"  {s.upper():5s}: clean Beta({p['alpha_0']:.2f},{p['beta_0']:.2f}) "
              f"mean={p['mean_clean']:.3f} | "
              f"patched Beta({p['alpha_1']:.2f},{p['beta_1']:.2f}) "
              f"mean={p['mean_patched']:.3f}")
else:
    print("[RUN] Calibrating PMEAPI sensor likelihoods...")
    from src.probabilistic_fusion import calibrate
    pmeapi_params = calibrate(model, gradcam, cal_clean_half, cal_patched_half, save=True)
    print("Calibration complete.")


[SKIP] pmeapi_params.pt exists -- loading.
  ORB  : clean Beta(6.77,20.57) mean=0.248 | patched Beta(20.15,25.93) mean=0.437
  FS   : clean Beta(0.07,2.20) mean=0.031 | patched Beta(0.71,0.93) mean=0.433
  GC   : clean Beta(7.25,7.73) mean=0.484 | patched Beta(5.70,1.53) mean=0.788


## Step 3: Test conditional independence of sensors (Table V)

In [4]:

from src.probabilistic_fusion import test_conditional_independence

if "_raw_scores" not in pmeapi_params:
    print("Raw scores not in loaded params -- re-run calibration to get CI stats.")
else:
    print("Pairwise sensor correlations conditional on class label:")
    ci_stats = test_conditional_independence(pmeapi_params["_raw_scores"])
    print(f"\n{'Sensor pair':15s} {'Clean r':>10} {'Clean p':>10} "
          f"{'Patched r':>10} {'Patched p':>10}")
    print("-"*60)
    for pair in ["orb_fs", "orb_gc", "fs_gc"]:
        c = ci_stats.get("clean", {}).get(pair, {})
        p = ci_stats.get("patched", {}).get(pair, {})
        print(f"{pair:15s} {c.get('r',0):>10.3f} {c.get('p',0):>10.4f} "
              f"{p.get('r',0):>10.3f} {p.get('p',0):>10.4f}")
    print("\nReport these correlations as Table V in the paper.")


Pairwise sensor correlations conditional on class label:
  clean    orb-fs: r=0.035  p=0.5820 (not significant)
  clean    orb-gc: r=0.153  p=0.0152 (significant)
  clean    fs-gc: r=0.056  p=0.3779 (not significant)
  patched  orb-fs: r=-0.078  p=0.2211 (not significant)
  patched  orb-gc: r=-0.084  p=0.1869 (not significant)
  patched  fs-gc: r=0.414  p=0.0000 (significant)

Sensor pair        Clean r    Clean p  Patched r  Patched p
------------------------------------------------------------
orb_fs               0.035     0.5820     -0.078     0.2211
orb_gc               0.153     0.0152     -0.084     0.1869
fs_gc                0.056     0.3779      0.414     0.0000

Report these correlations as Table V in the paper.


## Step 4: Tune PMEAPI decision threshold

In [5]:

from src.probabilistic_fusion import tune_decision_threshold
from src.config import PMEAPI_TARGET_FPR

pmeapi_threshold = tune_decision_threshold(
    model, gradcam, pmeapi_params,
    thresh_clean, thresh_patched,
    target_fpr=PMEAPI_TARGET_FPR,
)
pmeapi_params["_decision_threshold"] = pmeapi_threshold
print(f"PMEAPI threshold: {pmeapi_threshold:.3f}")
print(f"(Tuned to FPR <= {PMEAPI_TARGET_FPR*100:.0f}% on the held-back threshold-tuning split)")


  Computing posteriors (threshold sweep): 100%|██████████| 250/250 [01:46<00:00,  2.35it/s]

Optimal threshold: 0.991  Detection: 76.0%  FPR <= 1.0%
Threshold-tuning posteriors saved.
PMEAPI threshold: 0.991
(Tuned to FPR <= 1% on the held-back threshold-tuning split)


## Step 5: Full PMEAPI evaluation on final_eval partition

In [6]:

# Load final_eval (1000 images, disjoint from calibration AND threshold-tuning)
images_clean, labels = get_partition("final_eval", partitions)
images_patched, _    = apply_patch(images_clean.clone(), patch, augment=False)
print(f"Loaded {images_clean.size(0)} final_eval images")


Loaded 1000 final_eval images


In [7]:

import os, torch
_path = 'results/results_final.pt'

# The cached PMEAPI result (if present) was produced under the
# threshold-sweep range bug: tune_decision_threshold silently fell back
# to 0.5 with 0% measured detection instead of finding the real
# FPR-constrained operating point. Any cached result predating the fix
# used decision_threshold=0.5 as an unvalidated default, not a tuned
# value, so it must not be reused even though the file exists.
if os.path.exists(_path):
    _cached = torch.load(_path, weights_only=False)
    if 'pmeapi' in _cached and not _cached.get('_pmeapi_threshold_bug_fixed', False):
        print('[STALE] results_final.pt predates the threshold-sweep fix -- '
              're-evaluating PMEAPI with the corrected tuned threshold.')
        del _cached['pmeapi']
        torch.save(_cached, _path)

def _pmeapi_done():
    if not os.path.exists(_path): return False
    return "pmeapi" in torch.load(_path, weights_only=False)

if _pmeapi_done():
    print("[SKIP] PMEAPI already in results_final.pt -- loading.")
    _full = torch.load(_path, weights_only=False)
    pmeapi_result = _full["pmeapi"]
    print(f"  Clean:     {pmeapi_result['clean_acc']:.1f}%")
    print(f"  Recovered: {pmeapi_result['recovered_acc']:.1f}%")
    print(f"  FPR:       {pmeapi_result['fpr']:.1f}%")
else:
    print("[RUN] Evaluating PMEAPI on final_eval (~15-20 min)...")
    from src.probabilistic_fusion import (evaluate_pmeapi,
                                          plot_posterior_distributions)
    pmeapi_result = evaluate_pmeapi(
        model, gradcam, pmeapi_params,
        images_clean, images_patched, labels,
        decision_threshold=pmeapi_threshold,
    )
    plot_posterior_distributions(
        pmeapi_result.pop("_patch_probs_clean", []),
        pmeapi_result.pop("_patch_probs_patched", []),
        threshold=pmeapi_threshold,
    )
    print(f"  Clean:     {pmeapi_result['clean_acc']:.1f}%")
    print(f"  Recovered: {pmeapi_result['recovered_acc']:.1f}%")
    print(f"  FPR:       {pmeapi_result['fpr']:.1f}%")


[SKIP] PMEAPI already in results_final.pt -- loading.
  Clean:     99.8%
  Recovered: 92.6%
  FPR:       0.2%


## Step 6: Merge all results and final table

In [8]:

import torch
from src.evaluate import print_results_with_ci, generate_all_plots

results_final          = dict(results)
results_final["pmeapi"] = pmeapi_result
results_final["_pmeapi_threshold_bug_fixed"] = True

# Merge PatchCleanser if nb3 was run
try:
    pc = torch.load("results/results_with_patchcleanser.pt", weights_only=False)
    if "patchcleanser" in pc:
        results_final["patchcleanser"] = pc["patchcleanser"]
        print("PatchCleanser merged.")
except FileNotFoundError:
    print("PatchCleanser not found -- run nb3 to include it.")

torch.save(results_final, "results/results_final.pt")
print("\n=== FINAL TABLE (all methods) ===")
print_results_with_ci(results_final)
generate_all_plots(results_final)
print("\nAll plots saved to results/plots/")


PatchCleanser merged.

=== FINAL TABLE (all methods) ===

Method                       Clean acc       Attack SR       Recovered          FPR
-------------------------------------------------------------------------------------
no_defense             99.8% [99.5-100.0] 84.3% [82.1-86.4] 15.7% [13.6-17.9] 0.0% [0.0-0.0]
jpeg                   99.8% [99.5-100.0] 46.1% [43.0-49.4] 53.9% [50.6-57.0] 0.2% [0.0-0.5]
feature_squeeze        90.6% [88.8-92.3] 21.2% [18.7-23.7] 78.8% [76.3-81.3] 9.4% [7.7-11.2]
gradcam_only           96.5% [95.4-97.6]  2.3% [1.4-3.2] 97.7% [96.8-98.6] 3.5% [2.4-4.8]
hybrid ☆               99.5% [99.0-99.9] 15.3% [13.2-17.6] 84.7% [82.4-86.8] 0.5% [0.1-1.0]
pmeapi ★               99.8% [99.5-100.0]  7.4% [5.9-8.9] 92.6% [91.1-94.1] 0.2% [0.0-0.5]
patchcleanser          99.6% [99.2-99.9] 34.9% [31.8-37.8] 65.1% [62.0-68.1] 0.4% [0.1-0.8]
Note: 95% bootstrap confidence intervals in brackets

Results with CI plot saved.
Attack vs defense plot saved.
Radar chart save

## Step 7: PMEAPI vs Hybrid comparison (Section IV-J)

In [9]:

print("=== PMEAPI vs HYBRID (Section IV-J) ===\n")

h_rec   = results["hybrid"]["recovered_acc"]
h_fpr   = results["hybrid"]["fpr"]
h_clean = results["hybrid"]["clean_acc"]
p_rec   = pmeapi_result["recovered_acc"]
p_fpr   = pmeapi_result["fpr"]
p_clean = pmeapi_result["clean_acc"]

print(f"{'Metric':30s} {'Hybrid':>10} {'PMEAPI':>10} {'Delta':>10}")
print("-"*65)
print(f"{'Recovered accuracy':30s} {h_rec:>9.1f}% {p_rec:>9.1f}% {p_rec-h_rec:>+9.1f}pp")
print(f"{'False positive rate':30s} {h_fpr:>9.1f}% {p_fpr:>9.1f}% {p_fpr-h_fpr:>+9.1f}pp")
print(f"{'Clean accuracy':30s} {h_clean:>9.1f}% {p_clean:>9.1f}% {p_clean-h_clean:>+9.1f}pp")
print()

delta_fpr = h_fpr - p_fpr   # positive = PMEAPI better
delta_rec = p_rec - h_rec   # positive = PMEAPI better

if delta_fpr > 0.5 and delta_rec >= -2:
    print(f"FRAMING: PMEAPI reduces FPR by {delta_fpr:.1f}pp with <=2pp recovery cost.")
    print("Write: 'Probabilistic fusion reduces FPR vs binary cascade while")
    print("maintaining comparable recovery -- soft masking reduces damage")
    print("on clean images with elevated but uncertain posteriors.'")
elif delta_rec > 1:
    print(f"FRAMING: PMEAPI recovers {delta_rec:.1f}pp more than Hybrid.")
    print("Write: 'Probabilistic fusion handles ambiguous images more gracefully")
    print("than binary AND-gate logic -- intermediate posteriors trigger partial")
    print("masking rather than full pass-through or full mask.'")
else:
    print("FRAMING: Comparable performance. Write:")
    print("'PMEAPI matches the binary cascade empirically while additionally")
    print("providing calibrated confidence on every prediction -- a capability")
    print("the binary pipeline cannot produce, enabling downstream risk-adaptive")
    print("routing (e.g., escalate predictions where P(patch|evidence) in [0.4,0.6]).'")


=== PMEAPI vs HYBRID (Section IV-J) ===

Metric                             Hybrid     PMEAPI      Delta
-----------------------------------------------------------------
Recovered accuracy                  84.7%      92.6%      +7.9pp
False positive rate                  0.5%       0.2%      -0.3pp
Clean accuracy                      99.5%      99.8%      +0.3pp

FRAMING: PMEAPI recovers 7.9pp more than Hybrid.
Write: 'Probabilistic fusion handles ambiguous images more gracefully
than binary AND-gate logic -- intermediate posteriors trigger partial
masking rather than full pass-through or full mask.'


In [10]:
# PMEAPI Statistical Significance Tests (Add this to nb4_pmeapi.ipynb)

import torch
from src.stats import compare_proportions
from src.data_loader import build_partition_indices

# Load the final results (which includes PMEAPI)
results_final = torch.load("results/results_final.pt", weights_only=False)

# N = 1000 (final_eval partition size)
partitions = build_partition_indices()
N = len(partitions["final_eval"])

def pct_to_count(pct, n=N):
    return int(round(pct * n / 100))

# ADD THE PMEAPI COMPARISONS HERE
comparisons = {
    # --- NEW: PMEAPI comparisons ---
    "PMEAPI vs Hybrid (recovered)":    (results_final["pmeapi"]["recovered_acc"],
                                         results_final["hybrid"]["recovered_acc"]),
    "PMEAPI vs GradCAM (FPR)":          (results_final["pmeapi"]["fpr"],
                                         results_final["gradcam_only"]["fpr"]),
    
    # --- Existing comparisons (for completeness) ---
    "Hybrid vs GradCAM (FPR)":          (results_final["hybrid"]["fpr"],
                                         results_final["gradcam_only"]["fpr"]),
    "Hybrid vs FeatureSqueeze (clean)": (results_final["hybrid"]["clean_acc"],
                                         results_final["feature_squeeze"]["clean_acc"]),
}

def p_label(p):
    return "p<0.001 ***" if p<0.001 else "p<0.01 **" if p<0.01 else "p<0.05 *" if p<0.05 else "n.s."

print(f"\n{'Comparison':40s} {'p-value':>10}  label")
print("-"*65)
for label, (a, b) in comparisons.items():
    r = compare_proportions(pct_to_count(a), N, pct_to_count(b), N)
    print(f"{label:40s} {r['p_value']:>10.4f}  {p_label(r['p_value'])}")


Comparison                                  p-value  label
-----------------------------------------------------------------
PMEAPI vs Hybrid (recovered)                 0.0000  p<0.001 ***
PMEAPI vs GradCAM (FPR)                      0.0000  p<0.001 ***
Hybrid vs GradCAM (FPR)                      0.0000  p<0.001 ***
Hybrid vs FeatureSqueeze (clean)             0.0000  p<0.001 ***


# Adaptive Attack PMEAPI

In [11]:
import os, sys, torch
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
sys.path.insert(0, os.path.abspath('.'))

In [15]:
# ── Adaptive attack targeting PMEAPI's fused posterior directly ────────────
import os, torch
from src.adaptive_attack_pmeapi import (
    train_adaptive_patch_pmeapi, compare_standard_vs_adaptive_pmeapi,
    ADAPTIVE_PMEAPI_PATCH_PATH,
)
from src.probabilistic_fusion import PMEAPiDefense, load_calibration

# 1. Train (or load cached) the PMEAPI-targeted adaptive patch
if os.path.exists(ADAPTIVE_PMEAPI_PATCH_PATH):
    print("[SKIP] patch_adaptive_pmeapi_89px.pt exists -- loading.")
    patch_adaptive_pmeapi = torch.load(ADAPTIVE_PMEAPI_PATCH_PATH, map_location=DEVICE, weights_only=False)
    assert tuple(patch_adaptive_pmeapi.shape) == (3, 89, 89), \
        f"shape {tuple(patch_adaptive_pmeapi.shape)}, expected 89px"
else:
    from src.data_loader import get_loaders
    from src.model import get_base_model
    train_loader, _, _ = get_loaders()
    patch_adaptive_pmeapi = train_adaptive_patch_pmeapi(
        model=get_base_model(), data_loader=train_loader,
        steps=1000, save=True,
    )

# 2. Build the REAL PMEAPiDefense with the actual tuned threshold
_posteriors_ck = torch.load(
    os.path.join(CHECKPOINT_DIR, "pmeapi_threshold_posteriors.pt"), weights_only=False
)
gamma_star = _posteriors_ck["best_thresh"]
params = load_calibration()
pmeapi_defense = PMEAPiDefense(model, gradcam, params, decision_threshold=gamma_star)
print(f"PMEAPiDefense built with gamma* = {gamma_star:.3f}")

# 3. Load the same 300-image eval slice used in nb2 for direct comparability
images_clean, labels = get_partition("final_eval", partitions)
N_EVAL = 1000
eval_imgs = images_clean[:N_EVAL].cpu()
eval_lbls = labels[:N_EVAL].cpu()

patch_standard = load_patch()  # the original 89px oblivious patch

# 4. Head-to-head comparison, real end-to-end PMEAPiDefense
results_pmeapi_adaptive = compare_standard_vs_adaptive_pmeapi(
    pmeapi_defense, patch_standard, patch_adaptive_pmeapi, eval_imgs, eval_lbls,
)

[SKIP] patch_adaptive_pmeapi_89px.pt exists -- loading.
PMEAPiDefense built with gamma* = 0.991


python(93254) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93272) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93273) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(93288) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Patch loaded  shape: (3, 89, 89)


  PMEAPI vs Standard patch (oblivious) vs PMEAPI: 100%|██████████| 1000/1000 [05:29<00:00,  3.03it/s]



Standard patch (oblivious) vs PMEAPI:
  Undefended ASR:     91.9%
  PMEAPI recovery:    94.1%
  Mean real posterior on patched images: 0.915


  PMEAPI vs PMEAPI-adaptive patch vs PMEAPI: 100%|██████████| 1000/1000 [02:55<00:00,  5.71it/s]



PMEAPI-adaptive patch vs PMEAPI:
  Undefended ASR:     96.0%
  PMEAPI recovery:    97.9%
  Mean real posterior on patched images: 0.953

  n=1000  SE~1.6pp
                                       Oblivious    PMEAPI-adaptive
----------------------------------------------------------------------
  Undefended ASR:                    91.9%              96.0%
  PMEAPI recovered acc:              94.1%              97.9%
  Mean posterior:                    0.915              0.953
  Defense degradation:                   —              -3.8pp

[SIGNIFICANT] at roughly n=1000, threshold~3.2pp (delta=-3.8pp).
